In [4]:
import os
import sys
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from datasets import load_from_disk
from scipy.fftpack import fft

# 设置绘图风格
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False

# 自动选择设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Running on device: {device}")

# 导入你的自定义模块 (确保路径正确)
sys.path.append(os.getcwd()) 

# 尝试导入你的 Encoder
try:
    # 假设你的主类在 math_encoders.py 中，通常叫 TSVectorEncoder 或类似名字
    # 这里根据你的文件内容，我使用通用的加载逻辑，请根据实际类名修改
    from encoder.math_encoders import BalancedHybridEncoder as MyEncoder 
except ImportError:
    # 如果找不到特定类名，尝试加载整个模块供后续调用
    import math_encoders
    print("⚠️ 未找到 TSVecEncoder，请在下方代码块手动指定 Encoder 类名")

✅ Running on device: cuda


In [ ]:
# === 配置路径 ===
DATASET_PATH = "correction_datasets_double_res" # 参考 encoder_ana.ipynb 的路径
CKPT_PATH = "checkpoints/best_model.pth"     # 模型权重路径

# === 加载数据 ===
print("⏳ Loading datasets...")
try:
    dataset_dict = load_from_disk(DATASET_PATH)
    train_ds = dataset_dict['train']
    test_ds = dataset_dict['test']
    print(f"✅ Data Loaded. Train: {len(train_ds)}, Test: {len(test_ds)}")
except Exception as e:
    print(f"❌ Load failed: {e}")
    # Mock data strictly for code demonstration if file missing
    print("⚠️ Generating MOCK data for demonstration...")
    class MockDS:
        def __init__(self, n=1000): self.data = np.random.randn(n, 96).astype(np.float32)
        def __getitem__(self, idx): return {'past_values': self.data[idx], 'dataset_name': 'MOCK_DB'}
        def __len__(self): return len(self.data)
    train_ds = MockDS(2000)
    test_ds = MockDS(100)

# === 加载模型 ===
# 请将 'MyEncoder' 替换为你 math_encoders.py 中实际的类名
# 假设参数与你的训练配置一致
INPUT_DIM = 1
HIDDEN_DIM = 128
try:
    # 实例化模型 (根据 math_encoders.py 的实际 init 参数调整)
    encoder = math_encoders.TSVecEncoder(input_dim=INPUT_DIM, output_dim=HIDDEN_DIM).to(device)
    
    # 加载权重
    if os.path.exists(CKPT_PATH):
        encoder.load_state_dict(torch.load(CKPT_PATH, map_location=device))
        print(f"✅ Loaded checkpoint from {CKPT_PATH}")
    else:
        print(f"⚠️ Checkpoint not found at {CKPT_PATH}, using random weights.")
    
    encoder.eval()
except Exception as e:
    print(f"❌ Model init failed: {e}. Please check class name and args.")

⏳ Loading datasets...
❌ Load failed: Directory Datasets/processed_datasets is neither a `Dataset` directory nor a `DatasetDict` directory.
⚠️ Generating MOCK data for demonstration...
❌ Model init failed: name 'math_encoders' is not defined. Please check class name and args.


In [6]:
# === 预计算配置 ===
BATCH_SIZE = 256
DB_EMBS = []      # 存储主向量 (e.g., Shape/Trend)
DB_ERR_EMBS = []  # 存储误差向量 (如果有)
DB_RAW = []       # 存储原始序列用于绘图
DB_METADATA = []  # 存储来源数据集名称

print("🚀 Building Vector Database (Pre-computing)...")

def collate_fn(batch):
    # 简化的 collate，提取序列数据
    seqs = [b['past_values'] for b in batch]
    names = [b.get('dataset_name', 'unknown') for b in batch]
    return torch.tensor(np.array(seqs), dtype=torch.float32), names

# 创建 DataLoader
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

with torch.no_grad():
    for seqs, names in tqdm(train_loader, desc="Encoding DB"):
        seqs = seqs.to(device)
        
        # === 核心编码步骤 (参考 math_encoders.py) ===
        # 假设 encoder.encode 返回单个向量，或者返回 (emb, err_emb) 元组
        # 根据你的 dataset.py 中 combined_embs 和 combined_err_embs 的逻辑
        
        # 情况 A: 如果模型返回字典或元组
        outputs = encoder(seqs) 
        
        # 下面逻辑需要根据你的 math_encoders.py 实际返回值微调
        if isinstance(outputs, tuple):
            emb = outputs[0]
            err_emb = outputs[1] if len(outputs) > 1 else torch.zeros_like(emb)
        else:
            emb = outputs
            err_emb = torch.zeros_like(emb) # 如果没有第二路向量，则补零
            
        # 归一化以便快速计算 Cosine Similarity
        emb = F.normalize(emb, p=2, dim=1)
        err_emb = F.normalize(err_emb, p=2, dim=1)
        
        DB_EMBS.append(emb.cpu())
        DB_ERR_EMBS.append(err_emb.cpu())
        DB_RAW.append(seqs.cpu())
        DB_METADATA.extend(names)

# 拼接成大张量并转回 Device (如果显存够大，这样最快)
# 如果显存不够，保持在 CPU
DB_EMBS = torch.cat(DB_EMBS, dim=0).to(device)     # (N, Dim)
DB_ERR_EMBS = torch.cat(DB_ERR_EMBS, dim=0).to(device) # (N, Dim)
DB_RAW = torch.cat(DB_RAW, dim=0).numpy()          # (N, Seq_Len)
DB_METADATA = np.array(DB_METADATA)

print(f"✅ Database Built! Shape: {DB_EMBS.shape}")

🚀 Building Vector Database (Pre-computing)...


Encoding DB:   0%|          | 0/8 [00:00<?, ?it/s]


NameError: name 'encoder' is not defined

In [ ]:
def analyze_retrieval(query_idx, alpha, beta, top_k=5, source='test'):
    """
    params:
    - query_idx: 样本索引
    - alpha: 主向量相似度权重
    - beta: 辅助/误差向量相似度权重
    - source: 'test' or 'train'
    """
    # 1. 获取 Query 数据
    target_ds = test_ds if source == 'test' else train_ds
    raw_sample = target_ds[query_idx]
    
    # 兼容处理
    if isinstance(raw_sample, dict):
        seq_np = raw_sample['past_values']
        ds_name = raw_sample.get('dataset_name', 'Unknown')
    else:
        seq_np = raw_sample
        ds_name = "Unknown"
        
    query_seq = torch.tensor(seq_np, dtype=torch.float32).unsqueeze(0).to(device)
    
    # 2. 编码 Query
    with torch.no_grad():
        outputs = encoder(query_seq)
        if isinstance(outputs, tuple):
            q_emb, q_err_emb = outputs[0], outputs[1]
        else:
            q_emb, q_err_emb = outputs, torch.zeros_like(outputs)
            
        q_emb = F.normalize(q_emb, p=2, dim=1)
        q_err_emb = F.normalize(q_err_emb, p=2, dim=1)

    # 3. 计算相似度 (矩阵乘法)
    # Sim Shape: (1, N_DB)
    sim_main = torch.mm(q_emb, DB_EMBS.t())
    sim_err = torch.mm(q_err_emb, DB_ERR_EMBS.t())
    
    # === 关键：融合 Alpha 和 Beta ===
    # Score = alpha * Main_Sim + beta * Err_Sim
    total_score = alpha * sim_main + beta * sim_err
    
    # 4. 获取 Top-K
    scores, indices = torch.topk(total_score, k=top_k, dim=1)
    scores = scores.cpu().numpy()[0]
    indices = indices.cpu().numpy()[0]
    
    retrieved_raw = DB_RAW[indices]
    retrieved_meta = DB_METADATA[indices]
    
    # ==========================
    #        可视化展示
    # ==========================
    fig = plt.figure(figsize=(18, 10))
    gs = fig.add_gridspec(2, 3)

    # A. 时域波形对比 (Top-K Waveforms)
    ax1 = fig.add_subplot(gs[0, :2])
    x_axis = np.arange(len(seq_np))
    # 绘制邻居 (淡色)
    for i, seq in enumerate(retrieved_raw):
        ax1.plot(x_axis, seq, color='gray', alpha=0.4, linewidth=1, label='Neighbor' if i==0 else "")
    # 绘制 Query (高亮)
    ax1.plot(x_axis, seq_np, color='red', linewidth=2.5, label=f'Query (Src: {ds_name})')
    ax1.set_title(f'Top-{top_k} Retrieved Neighbors (Alpha={alpha}, Beta={beta})', fontsize=14)
    ax1.legend()
    
    # B. 相似度得分分布 (Score Stats)
    ax2 = fig.add_subplot(gs[0, 2])
    ax2.barh(range(top_k), scores[::-1], color='skyblue')
    ax2.set_yticks(range(top_k))
    ax2.set_yticklabels([f"Rank {top_k-i}" for i in range(top_k)])
    ax2.set_xlabel("Weighted Score")
    ax2.set_title("Top-K Similarity Scores")
    
    # C. 频域分析 (FFT Spectrum)
    ax3 = fig.add_subplot(gs[1, 0])
    # Query FFT
    yf_q = fft(seq_np)
    xf = np.linspace(0.0, 1.0/(2.0), len(seq_np)//2)
    ax3.plot(xf, 2.0/len(seq_np) * np.abs(yf_q[0:len(seq_np)//2]), 'r-', linewidth=2, label='Query Freq')
    # Neighbor FFT Mean
    neigh_fft_sum = np.zeros_like(np.abs(yf_q[0:len(seq_np)//2]))
    for seq in retrieved_raw:
        yf = fft(seq)
        neigh_fft_sum += 2.0/len(seq) * np.abs(yf[0:len(seq)//2])
    ax3.plot(xf, neigh_fft_sum / top_k, 'k--', alpha=0.7, label='Avg Neighbor Freq')
    ax3.set_title("Frequency Domain Consistency")
    ax3.legend()

    # D. 领域成分统计 (Domain Composition)
    ax4 = fig.add_subplot(gs[1, 1])
    unique_domains, counts = np.unique(retrieved_meta, return_counts=True)
    ax4.pie(counts, labels=unique_domains, autopct='%1.1f%%', startangle=140, colors=sns.color_palette("pastel"))
    ax4.set_title("Source Domain Composition")

    # E. 统计指标文本
    ax5 = fig.add_subplot(gs[1, 2])
    ax5.axis('off')
    # 计算一些简单的统计量
    mse_dist = np.mean((retrieved_raw - seq_np) ** 2, axis=1)
    text_str = f"--- Statistics ---\n\n"
    text_str += f"Avg Score: {np.mean(scores):.4f}\n"
    text_str += f"Max Score: {np.max(scores):.4f}\n"
    text_str += f"Avg MSE (Dist): {np.mean(mse_dist):.4f}\n\n"
    text_str += f"--- Parameter Info ---\n"
    text_str += f"Alpha (Main): {alpha}\n"
    text_str += f"Beta (Aux/Err): {beta}\n"
    
    ax5.text(0.1, 0.5, text_str, fontsize=14, family='monospace')

    plt.tight_layout()
    plt.show()

In [ ]:
# ==========================================
#        🧪 实验参数设置 (在此处修改)
# ==========================================

# 控制参数：根据你的假设调节权重
# 如果 Alpha > Beta，更看重主特征（如形状、趋势）
# 如果 Beta 增大，可能更看重细节误差匹配或特定子特征
ALPHA = 1.0   
BETA = 0.5    

# 检索配置
TOP_K = 10
SAMPLE_ID = 42      # 随机选择一个索引
SOURCE = 'test'     # 'test' 或 'train'

# ==========================================
#            执行可视化
# ==========================================
analyze_retrieval(
    query_idx=SAMPLE_ID, 
    alpha=ALPHA, 
    beta=BETA, 
    top_k=TOP_K, 
    source=SOURCE
)